# Busca por termos relacionados a IA

Ideia geral: 
- Preparar batches de dias e disparar buscas assíncronas usando diferentes processos. 
- Cada processo é responsável por coletar notícias do seu dia.


In [1]:
from dateutil import parser as dateparser
from dateutil.relativedelta import relativedelta
from datetime import datetime, timedelta
from multiprocessing import Pool

In [2]:
general_kw = [
    # AI-related keywords
    "artificial intelligence",
    "generative AI",
    "deep learning",
    "large language models",
    "llms",
    "autonomous agents",
    # actors
    "openai",
    "google",
    "microsoft",
    "anthropic",
    # social impact implications
    "job displacement",
    "digital divide",
    "algorithmic bias",
    # political implications
    "AI regulation",
    "AI ethics",
    "AI investment",
    ]

In [3]:
day_batch = 10
max_news_per_day = 100
increment = timedelta(days=1)
start_date = datetime(2026, 3, 1)
end_date = datetime(2026, 4, 1)

In [4]:
# agrupar dias em batches de 10

In [5]:
from gnews import GNews

def fetch_news_for_target_date(target_date: datetime, key_word: str) -> list[dict]:
    start_date = target_date - increment
    end_date = target_date

    client = GNews()
    client.language = 'en'
    client.max_results = max_news_per_day
    client.start_date = (start_date.year, start_date.month, start_date.day)
    client.end_date = (end_date.year, end_date.month, end_date.day)

    try: 
        news = client.get_news(key_word)
    except Exception as e:        
        print(f"Error fetching news for {key_word} on {target_date.strftime('%Y-%m-%d')}: {e}")
        news = []
        
    return news

In [6]:
results = []

for kw in general_kw:
    print(f"Buscando notícias para a palavra-chave: {kw}")

    # reseting start dates and batches for each keyword
    curr_batch = []
    curr_date = start_date
    while curr_date < end_date:
        

        curr_batch.append(curr_date)
        curr_date = curr_date + increment

        if len(curr_batch) == day_batch:
            # disparar processos
            print(f"- Tentando buscar notícias para o periodo de {curr_batch[0].date()} a {curr_batch[-1].date()}")
            
            with Pool(processes=day_batch) as pool:
                curr_results = pool.starmap(fetch_news_for_target_date, [(date, kw) for date in curr_batch])
            print("\t - Busca concluída para o periodo")

            # adding the use kw for logs
            for batch_results in curr_results:
                for news in batch_results:
                    news['keyword'] = kw

            results.extend(curr_results)

            curr_batch = []
            # TODO: serializar aqui para evitar perda de dados

    # talvez sobre uns dias no final

Buscando notícias para a palavra-chave: artificial intelligence
- Tentando buscar notícias para o periodo de 2026-03-01 a 2026-03-10
	 - Busca concluída para o periodo
- Tentando buscar notícias para o periodo de 2026-03-11 a 2026-03-20
	 - Busca concluída para o periodo
- Tentando buscar notícias para o periodo de 2026-03-21 a 2026-03-30
	 - Busca concluída para o periodo
Buscando notícias para a palavra-chave: generative AI
- Tentando buscar notícias para o periodo de 2026-03-01 a 2026-03-10
	 - Busca concluída para o periodo
- Tentando buscar notícias para o periodo de 2026-03-11 a 2026-03-20
Error fetching news for generative AI on 2026-03-17: Failed to fetch or parse news feed: HTTPSConnectionPool(host='news.google.com', port=443): Max retries exceeded with url: /rss/articles/CBMi3gFBVV95cUxPUVR1OGw2YjBtdzZLejhWZ0daNGROUTV2LTh1UGJQcUZnQ3lLY1dWSEswVU5saDZ5TWJyZ1lCNS1lOFRoaXFIZE1RNEM5dkZJbFgtOWJEaHB3VktwVG44bDdfVG1fSlNuNTM3bThYd2JJTmxXS1RKWXRSNVI4YXVZQVFMdENUdWlObDBOU1poS1MyTDlQTnJN

In [7]:
unpacked_results = [news for batch in results for news in batch]

In [8]:
len(unpacked_results)

26672

In [9]:
expected_num_news = len(general_kw) * max_news_per_day * ((end_date - start_date).days)
print(f"Expected number of news: {expected_num_news}")

Expected number of news: 49600


# Visualizando algumas noticias coletadas

In [10]:
import random

In [11]:
samples = 10

for i in range(samples):
    ex = random.choice(unpacked_results)

    for key, value in ex.items():
        print(f"{key}: {value}")

    print("\n" + "-"*50 + "\n")

title: The world’s most valuable company just sent another signal that AI agents are going to be everywhere | CNN Business - CNN
description: The world’s most valuable company just sent another signal that AI agents are going to be everywhere | CNN Business  CNN
published date: Mon, 16 Mar 2026 07:00:00 GMT
url: https://news.google.com/rss/articles/CBMic0FVX3lxTE1WeTFfT240c3JqYU8wMm81U3pTZlJibm4tYTRsMWJvRkVkNlY4UzBRUVFidDVabHFrMGpTcVJULVFPMTRUQWNZbzljTl94dXkwanlqT2JmNEdidXU2NlpNalNCQS1TNk1sblNDaXdIVHp2MVU?oc=5&hl=en-US&gl=US&ceid=US:en
publisher: {'href': 'https://www.cnn.com', 'title': 'CNN'}
keyword: artificial intelligence

--------------------------------------------------

title: Nvidia’s Inference Power Play: Outmaneuvering Google And Amazon - Trefis
description: Nvidia’s Inference Power Play: Outmaneuvering Google And Amazon  Trefis
published date: Thu, 19 Mar 2026 07:00:00 GMT
url: https://news.google.com/rss/articles/CBMivwFBVV95cUxPNTZobGhPT0pOQnRXMlNVQWRXcllTQUp6UUNCOXdzSU9E

# Serializando as noticias brutas

In [12]:
import pickle

In [13]:
pickle.dump(unpacked_results, open("../raw/gnews_results_mar_abril_ai_2026.pkl", "wb"))